# Implementation: One Pixel Attack for Fooling Deep Neural Networks

### Environment

This Jupyter Notebook is being run on Python version 3.14.4 with dependencies listed in the requirements.txt file.

### Imports

In [ ]:
#general
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_MIN_LOG_LEVEL"] = "3"
from pathlib import Path
import random
import json
from collections import OrderedDict

#machine learning
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import keras
from keras import layers, models
from keras.datasets import cifar10
import numpy as np
import tensorflow as tf
import seaborn as sns
print(tf.config.list_physical_devices("GPU"))

## Section 1: Model Implementations

The models first need to be implemented, in the way they are structured in the paper so that we can create both a training function, to produce a `.keras` file to load the model trained on the CIFAR-10 data, and a processing function for each model, taking in a list of image files and returning their final classified label, as well as class probabilities.

In [ ]:
class_names = ['airplane', 
                'automobile', 
                'bird', 
                'cat', 
                'deer', 
                'dog', 
                'frog', 
                'horse', 
                'ship', 
                'truck']

### VGG16

Using the official implementation guide by [Keras](https://keras.io/api/applications/#usage-examples-for-image-classification-models) to train the VGG16 on the ``train/`` sample of the CIFAR-10 dataset.

In [ ]:
def train_model():
    (x_train, y_train), (x_test, y_test) = cifar10.load_data()
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    y_train = y_train.flatten()
    y_test = y_test.flatten()

    VGG16_model = models.Sequential([
        layers.Input(shape=(32,32,3)), #cifar-10 is 3 channel rgb 32x32 images

        #chunk 1: 64 neurons
        layers.Conv2D(64, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2,2), strides=2),

        #chunk 2: 128 neurons
        layers.Conv2D(128, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2,2), strides=2),

        #chunk 3: 256 neurons
        layers.Conv2D(256, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3,3), strides=1, padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=(2,2), strides=2),

        #chunk 4: 512 neurons
        layers.Conv2D(512, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(512, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(512, (3,3), strides=1, padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=(2,2), strides=2),

        #chunk 5: 512 neurons (again)
        layers.Conv2D(512, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(512, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(512, (3,3), strides=1, padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=(2,2), strides=2),

        #classification layers
        layers.Flatten(),
        layers.Dense(2048, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(2048, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(len(class_names), activation="softmax")
    ])

    VGG16_model.compile(
        optimizer = "adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    VGG16_model.fit(
        x_train,
        y_train,
        epochs=15,
        batch_size=64,
        validation_split=0.2
    )
    VGG16_model.save("VGG16.keras")   

def VGG16_function(images) :
    #predict a label
    x = np.asarray(images, dtype=np.float32)
    x = x / 255.0

    #predict the label
    predictions = VGG16_model.predict(x, verbose=0)

    results = []
    for prediction in predictions:
        predicted_index = np.argmax(prediction)
        predicted_label = class_names[predicted_index]

        #get class probabilities
        class_probabilities = {}
        for class_name, probability in zip(class_names, prediction):
            class_probabilities[class_name] = float(probability)

        #check the accuracy and return
        results.append({
            "predicted_label": predicted_label,
            "class_probabilities": class_probabilities
        })
    return results

# if model doesn't exist, train it, then load it
if os.path.exists("VGG16.keras") == False:
    train_model()
VGG16_model = keras.models.load_model("VGG16.keras")


### AllConv

In [ ]:
def train_allConv_model():
    (x_train, y_train), (x_test, y_test) = cifar10.load_data()
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    y_train = y_train.flatten()
    y_test = y_test.flatten()

    allConv_model = models.Sequential([

        layers.Input(shape=(32,32,3)),        
        
        #chunk 1: 96 neurons
        layers.Conv2D(96, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(96, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(96, (3,3), strides=2, padding="same", activation="relu"),
        layers.BatchNormalization(),

        #chunk 2: 192 neurons
        layers.Conv2D(192, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(192, (3,3), strides=1, padding="same", activation="relu"),

        layers.Dropout(0.3),

        layers.Conv2D(192, (3,3), strides=2, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(192, (3,3), strides=2, padding="same", activation="relu"),
        layers.BatchNormalization(),

        layers.Conv2D(192, (1,1), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(192, (1,1), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),

        #chunk 3 : 10 neurons
        layers.Conv2D(10, (1,1), strides=1, padding="same", activation="relu"),

        layers.AveragePooling2D((6,6), strides=1, padding="same"),

        #classification layers
        layers.Flatten(),
        layers.Dense(len(class_names), activation="softmax")

    ])

    allConv_model.compile(
        optimizer = "adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    allConv_model.fit(
        x_train,
        y_train,
        epochs=15,
        batch_size=64,
        validation_split=0.2
    )

    allConv_model.save("allConv.keras")   

def allConv_function(images) :
    #predict a label
    x = np.asarray(images, dtype=np.float32)
    x = x / 255.0

    #predict the label
    predictions = allConv_model.predict(x, verbose=0)

    results = []
    for prediction in predictions:
        predicted_index = np.argmax(prediction)
        predicted_label = class_names[predicted_index]

        #get class probabilities
        class_probabilities = {}
        for class_name, probability in zip(class_names, prediction):
            class_probabilities[class_name] = float(probability)

        #check the accuracy and return
        results.append({
            "predicted_label": predicted_label,
            "class_probabilities": class_probabilities
        })
    return results

# if model doesn't exist, train it, then load it
if os.path.exists("allConv.keras") == False:
    train_allConv_model()
allConv_model = keras.models.load_model("allConv.keras")

### NiN

In [ ]:
def train_NiN_model():
    (x_train, y_train), (x_test, y_test) = cifar10.load_data()
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    y_train = y_train.flatten()
    y_test = y_test.flatten()

    NiN_model = models.Sequential([

        layers.Input(shape=(32,32,3)),
        
        layers.Conv2D(192, (5,5), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(160, (1,1), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(96, (1,1), strides=1, padding="same", activation="relu"),

        layers.MaxPooling2D(pool_size=(3,3), strides=2),
        layers.Dropout(0.5),

        layers.Conv2D(192, (5,5), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(192, (5,5), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(192, (5,5), strides=1, padding="same", activation="relu"),
        
        layers.AveragePooling2D(pool_size=(3,3), strides=2),
        layers.Dropout(0.5),

        layers.Conv2D(192, (3,3), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(192, (1,1), strides=1, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(10, (1,1), strides=1, padding="same", activation="relu"),

        layers.AveragePooling2D(pool_size=(7,7), strides=1, padding="same"),
        layers.Dropout(0.5),

        #classification layers
        layers.Flatten(),
        layers.Dense(len(class_names), activation="softmax")

    ])

    NiN_model.compile(
        optimizer = "adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    NiN_model.fit(
        x_train,
        y_train,
        epochs=15,
        batch_size=64,
        validation_split=0.2
    )
    NiN_model.save("NiN.keras")   

def NiN_function(images) :
    #predict a label
    x = np.asarray(images, dtype=np.float32)
    x = x / 255.0

    #predict the label
    predictions = NiN_model.predict(x, verbose=0)

    results = []
    for prediction in predictions:
        predicted_index = np.argmax(prediction)
        predicted_label = class_names[predicted_index]

        #get class probabilities
        class_probabilities = {}
        for class_name, probability in zip(class_names, prediction):
            class_probabilities[class_name] = float(probability)

        #check the accuracy and return
        results.append({
            "predicted_label": predicted_label,
            "class_probabilities": class_probabilities
        })
    return results

# if model doesn't exist, train it, then load it
if os.path.exists("NiN.keras") == False:
    train_NiN_model()
NiN_model = keras.models.load_model("NiN.keras")

## Section 2: Dataset Pre-processing
Choosing an image at random from the `test` sample of the CIFAR-10 dataset, using an example model, VGG16 to classify it (detailing classification label and image index), and stopping once reached the total amount of input images (`TOTAL_INPUT_IMGS`).

In [ ]:
#global variables
TOTAL_INPUT_IMGS = 500

def preprocess():
    #loading the CIFAR-10 dataset, and getting a random sample
    (x_train, y_train), (x_test, y_test) = cifar10.load_data()
    dataset_indexes = random.sample(range(len(x_test)), TOTAL_INPUT_IMGS)

    # classifying all images
    predictions = VGG16_function(x_test[dataset_indexes])

    # saving original classifications
    images_found = []
    for image_index, prediction in zip(dataset_indexes, predictions):
        images_found.append({
            "image_index": image_index,
            "original_label": prediction["predicted_label"]
        })

    # writing output to a json file
    with open("chosen_images.json", "w") as file:
        json.dump(images_found, file, indent=4)

# if chosen_images.json doesn't exist, generate it
if os.path.exists("chosen_images.json") == False:
    preprocess()

## Section 3: Pixel perturbations Optimisation

### Untargeted Attack
- Fitness is measured as misclassification
- For each model/image/peturbation count combination:
    - Initialise the population with 400 candidates (e.g. [[x, y, R, G, B]] or [[x1, y1, R1, G1, B1], [x2, y2, R2, G2, B2], ...])
    - Evaluate fitness on initial population
    - Repeat:
        - Mutate children: each candidate is assigned 3 random members of the population to produce its child (population size 400->800)
        - Evaluate fitness on previous and current populations, 
        - Selection: keep children if better performing than its parent (population size 800->400)
    - **Stop when** misclassification, or exceeds 100 generations
    - Save best fitness to a json file

### Targeted Attack
- Fitness is measured as a class probability %
- For each model/image/peturbation count combination:
    - Initialise the population with 400 candidates (e.g. [[x, y, R, G, B]] or [[x1, y1, R1, G1, B1], [x2, y2, R2, G2, B2], ...])
    - Evaluate fitness on initial population
    - Repeat:
        - Mutate children: each candidate is assigned 3 random members of the population to produce its child (population size 400->800)
        - Evaluate fitness on previous and current populations, 
        - Selection: keep children if better performing than its parent (population size 800->400)
    - **Stop when** target class probability exceeds 50% or exceeds 100 generations
    - Save best fitness to a json file



In [ ]:
#global variables

MODEL_FUNCTIONS = {"VGG16" : VGG16_function,
                   "AllConv" : allConv_function,
                   "NiN": NiN_function}
# untargeted
UNTARGETED_PERTURBATION_COUNTS = [1,3,5]
UNTARGETED_POPULATION_SIZE = 400
UNTARGETED_MAX_GENERATIONS = 100
# targeted
TARGETED_PERTURBATION_COUNTS = [1,3,5]
TARGETED_POPULATION_SIZE = 300
TARGETED_MAX_GENERATIONS = 50
TARGET_CONFIDENCE = 0.5


class Candidate:
    def __init__(self, config, parent):
         self.config = config
         self.parent = parent
         self.fitness = None
         self.predicted_label = None

#returns a list of candidate objects
def initialise_population(population_size, image_shape, perturbation_count):
    # generating a randomised candidate
    candidates = []
    for _ in range(0, population_size):
        config = []
        for _ in range(0, perturbation_count):
            random_coord = [random.randint(0,image_shape[0] -1), random.randint(0,image_shape[1] -1)]
            # adding a randomised number for each colour channel
            for _ in range(image_shape[2]):
                random_coord.append(max(0, min(255, round(random.gauss(128,127)))))
            config.append(random_coord)
        candidates.append(Candidate(config, None))
    return candidates

def evaluate_fitness(population, model_function, image, target_class, original_label):
        #applying candidate perturbation to the image, and classifying
        perturbed_images = []
        for candidate in population:
            perturbed_image = image.copy()
            for perturbation in candidate.config:
                y, x, r, g, b = perturbation
                perturbed_image[y, x] = [r,g,b]
            perturbed_images.append(perturbed_image)
        perturbed_images = np.asarray(perturbed_images)
        predictions = model_function(perturbed_images)
        for candidate, result in zip(population, predictions):
            candidate.predicted_label = result["predicted_label"]
            if target_class is not None:
                candidate.fitness = float(result["class_probabilities"][target_class])
            else:
                candidate.fitness = 1 - float((result["class_probabilities"][original_label]))

#returns parents_and_children dictionary
def mutate_children(population, image_shape):
    F = 0.5
    children = []
    for i, candidate in enumerate(population):
        parents = []
        while len(parents) <3:
            parent_index = random.randrange(len(population))
            if parent_index != i and parent_index not in parents:
                parents.append(parent_index)
        config1 = np.array(population[parents[0]].config, dtype=float)
        config2 = np.array(population[parents[1]].config, dtype=float)
        config3 = np.array(population[parents[2]].config, dtype=float)
        child_config = config1 + F * (config2 - config3)
        fixed_config = []
        for perturbation in child_config:
            y, x, r, g, b = perturbation
            y = int(np.clip(round(y), 0, image_shape[0] - 1))
            x = int(np.clip(round(x), 0, image_shape[1] - 1))
            r = int(np.clip(round(r), 0, 255))
            g = int(np.clip(round(g), 0, 255))
            b = int(np.clip(round(b), 0, 255))
            fixed_config.append([y, x, r, g, b])
        children.append(Candidate(fixed_config, candidate))
    return children

# returns a list of candidate objects
def selection(population):
    half = len(population) // 2
    parents, children = population[:half], population[half:]
    new_population = []
    for parent, child in zip(parents, children):
        winner = None
        if child.fitness > parent.fitness:
            winner = child
        else:
            winner = parent
        winner.parent = None
        new_population.append(winner)
    return new_population

def pixel_perturbations(target_type):
    #loading pre-defined variables
    if target_type == "untargeted":
        perturbation_counts = UNTARGETED_PERTURBATION_COUNTS
        population_size = UNTARGETED_POPULATION_SIZE
        max_generations = UNTARGETED_MAX_GENERATIONS
        
    else:
        perturbation_counts = TARGETED_PERTURBATION_COUNTS
        population_size = TARGETED_POPULATION_SIZE
        max_generations = TARGETED_MAX_GENERATIONS
    #loading the CIFAR-10 dataset
    (x_train, y_train), (x_test, y_test) = cifar10.load_data()
    #loading the randomised and pre-guessed sample
    with open("chosen_images.json", "r") as file:
        images_found = json.load(file)
    # for each model
    for model_name, model_function in MODEL_FUNCTIONS.items():
        results = {}
        results[model_name] = {}
        #image progress
        image_progress = tqdm(
            enumerate(images_found),
            total=len(images_found),
            desc= target_type + " attack using " + model_name,
            unit="image"
        )
        #loop for each image
        for image_index_loop, image_data in image_progress:
            image_index = image_data["image_index"]
            image = x_test[image_index]
            original_label = image_data["original_label"]
            results[model_name][str(image_index)] = {}
            target_classes = []
            if target_type == "targeted":
                for a_class in class_names:
                    if a_class != original_label:
                        target_classes.append(a_class)
            else:
                target_classes = [None]
            #loop for each perturbation count
            for perturbation_count in perturbation_counts:
                if target_type == "targeted":
                    results[model_name][str(image_index)][str(perturbation_count)] = {}
                #loop for each target class
                for target_index, target_class in enumerate(target_classes):
                    if target_class is None:
                        target_text = "any"
                    else:
                        target_text = str(target_index + 1) + "/" + str(len(target_classes))
                    winner = None
                    generations = 0
                    population = initialise_population(population_size, image.shape, perturbation_count)
                    evaluate_fitness(population, model_function, image, target_class, original_label)
                    # looping from 1-400 generations until found
                    while winner is None and generations < max_generations:
                        # updating progress bar
                        image_progress.set_postfix(OrderedDict([
                            ("image", str(image_index_loop+1) + "/" + str(len(images_found))),
                            ("perturbations",  str(perturbation_count) + "/(" + str(perturbation_counts)+")"),
                            ("target class",  target_text),
                            ("generation", str(generations + 1) +"/" + str(max_generations))
                        ]), refresh=True)
                        #mutate children
                        children = mutate_children(population, image.shape)
                        evaluate_fitness(children, model_function, image, target_class, original_label)
                        population += children
                        # check for successful candidate
                        for candidate in population:
                            if target_class is None:
                                if candidate.predicted_label != original_label:
                                    winner = candidate
                                    break
                            elif candidate.fitness > TARGET_CONFIDENCE:
                                winner = candidate
                                break
                        if winner is not None: break
                        #selection
                        population = selection(population)
                        #iterate
                        generations+=1
                    # failed to attack
                    best = winner if winner is not None else max(population, key = lambda c: c.fitness)
                    if target_class is None:
                        success = winner is not None
                    else:
                        success = best.predicted_label == target_class
                    result = {
                        "original_label" : original_label,
                        "attack_success": success,
                        "new_label" : best.predicted_label,
                        "config": best.config,
                        "confidence" : best.fitness
                    }
                    if target_class is None:
                        results[model_name][str(image_index)][str(perturbation_count)] = result
                    else:
                        results[model_name][str(image_index)][str(perturbation_count)][str(target_class)] = result 
        # write to file
        with open(model_name+"_"+target_type+"_results.json", "w") as file:
            json.dump(results, file, indent=4)

# if results don't exist, generate them
for attack_type in ["untargeted", "targeted"]:
    for model_name in MODEL_FUNCTIONS:
        filename = model_name + "_" + attack_type + "_results.json"
        if os.path.exists(filename) == False:
            pixel_perturbations(attack_type)
            break


## Section 4: Accuracy Evaluation


In [ ]:
# extracts the attack success and confidence values from JSON data
def extract_values(data):

    # initialises dictionaries to store results separately for each attack
    results = {
        '1': {"attack_successes": [], "confidences": []},
        '3': {"attack_successes": [], "confidences": []},
        '5': {"attack_successes": [], "confidences": []},
    }

    # recursively traverses JSON structure to find attack success and confidence values
    def recursive_extract(obj, attack_type=None):

        # checks whether current object is a dictionary
        if isinstance(obj, dict):

            # checks whether dictionary contains attack results and if it belongs to one of the configurations
            if "attack_success" in obj and attack_type in results:

                #stores attack success value
                results[attack_type]["attack_successes"].append(
                    obj["attack_success"]
                )

                # stores corresponding confidence value
                results[attack_type]["confidences"].append(
                    obj["confidence"]
                )
                return

            # iterates through all key-value pairs in dictionary
            for key, value in obj.items():

                #if key is from 1,3 or 5 update attack_type to keep track
                if str(key) in results:
                    recursive_extract(value, str(key))

                # else continue traversing in the current configuration
                else:
                    recursive_extract(value, attack_type)

        # checks if current object is a list
        elif isinstance(obj, list):
            
            # iterate through each item in list and continue recursive search
            for item in obj:
                recursive_extract(item, attack_type)

    #start traversing from top level of JSON data
    recursive_extract(data)

    # return all extracted results             
    return results


# calculates and prints different evaluation stats for each attack
def print_values(attack_tar, model, results):
    
    # iterates through results for 1, 3 and 5 attacks
    for attack_type, values in results.items():
        
        #retrieve lists 
        atk_suc = values["attack_successes"]
        conf = values["confidences"]

        print(f"\n{attack_tar} {model} - Attack: {attack_type}")

        print("Total samples:", len(atk_suc))
        print("Successful attacks:", sum(atk_suc))
        print("Attack success rate:", np.mean(atk_suc))
        print("Average confidence:", np.mean(conf))


# loads appropriate JSON file and calculates stats
def calc_values(attack_tar, model):

    if attack_tar == "untargeted":
        filename = "untargeted_results.json"

        with open(filename, "r") as f:
            data = json.load(f)
        
        # all untargeted attacks in same file, so selects specific model results
        model_data = data[model]

        results = extract_values(model_data)
        print_values(attack_tar, model, results)
    
    else:
        filename = f"{model}_targeted_results.json"

        with open(filename, "r") as f:
            data = json.load(f)

        results = extract_values(data)
        print_values(attack_tar, model, results)


# evaluates attacks for each model separately
for a in ["untargeted", "targeted"]:
    for m in ["VGG16", "AllConv", "NiN"]:
        calc_values(a,m)




## Section 5: Visualisations

In [ ]:
def extract_class(data):

    # initialises dictionaries to store classes separately for each attack
    results = {
        '1': {"original_classes": [], "target_classes": []},
        '3': {"original_classes": [], "target_classes": []},
        '5': {"original_classes": [], "target_classes": []},
    }

    # recursively traverses JSON structure to find class label
    def recursive_extract_class(obj, attack_type=None):

        # checks whether current object is a dictionary
        if isinstance(obj, dict):

            # checks whether dictionary contains attack results and if it belongs to one of the configurations
            if "original_label" in obj and attack_type in results:

                if obj["attack_success"] == True:

                    #stores original class of image
                    results[attack_type]["original_classes"].append(
                        obj["original_label"]
                    )

                    # stores new class of label
                    results[attack_type]["target_classes"].append(
                        obj["new_label"]
                    )
                return

            # iterates through all key-value pairs in dictionary
            for key, value in obj.items():

                #if key is from 1,3 or 5 update attack_type to keep track
                if str(key) in results:
                    recursive_extract_class(value, str(key))

                # else continue traversing in the current configuration
                else:
                    recursive_extract_class(value, attack_type)

        # checks if current object is a list
        elif isinstance(obj, list):
            
            # iterate through each item in list and continue recursive search
            for item in obj:
                recursive_extract_class(item, attack_type)

    #start traversing from top level of JSON data
    recursive_extract_class(data)

    # return all extracted results             
    return results

def print_class(model, results):
    
    # iterates through results for 1, 3 and 5 attacks
    for attack_type, values in results.items():
        
        #retrieve lists 
        org_cls = values["original_classes"]
        tgt_cls = values["target_classes"]

        print(f"\n{model} - Attack: {attack_type}")
        print(org_cls)
        print(tgt_cls)


def calc_classes(attack_type, model):

    if attack_type == "untargeted":
        filename = "untargeted_results.json"

        with open(filename, "r") as f:
            data = json.load(f)
        
        # all untargeted attacks in same file, so selects specific model results
        model_data = data[model]

        results = extract_class(model_data)
        print_class(model, results)
    
    else:
        filename = f"{model}_targeted_results.json"

        with open(filename, "r") as f:
            data = json.load(f)

        results = extract_class(data)
        print_class(model, results)
    
    num_results = class_to_num(results)

    return num_results

def class_to_num(results):

    # maps classes names to numbers
    class_mapping = {
        "airplane": 0,
        "automobile": 1,
        "bird": 2,
        "cat": 3,
        "deer": 4,
        "dog": 5,
        "frog": 6,
        "horse": 7,
        "ship": 8,
        "truck": 9
    }

    # goes through each key pair in dict
    for key, values in results.items():

        org_cls = values["original_classes"]
        tgt_cls = values["target_classes"]

        org_cls_num = [class_mapping[i] for i in org_cls]
        tgt_cls_num = [class_mapping[j] for j in tgt_cls]

        values["org_cls_num"] = org_cls_num
        values["tgt_cls_num"] = tgt_cls_num

        print(org_cls_num)
        print(tgt_cls_num)

    return results


# plot bar graph of number of successful attacks for a specific class as the original and target class
def plot_original_target_classes(results, model):    

    classes = np.arange(10)

    # create figure with three subplots
    fig, axes = plt.subplots(1, 3, figsize=(16,5))

    for ax, (attack_type, values) in zip(axes, results.items()):

        org_cls_num = values["org_cls_num"]
        tgt_cls_num = values["tgt_cls_num"]

        # count occurrences of each original class
        org_count = np.bincount(org_cls_num, minlength=10)

        # count occurrences of each target class
        tgt_count = np.bincount(tgt_cls_num, minlength=10)

        width = 0.4

        # plot original classes (purple)
        ax.bar(
            classes - width/2,
            org_count,
            width,
            color="purple",
            label="Original Class"
        )

        # plot original classes (green)
        ax.bar(
            classes + width/2,
            tgt_count,
            width,
            color="green",
            label="Target Class"
        )

        ax.set_xticks(classes)
        ax.set_xlabel("Class")
        ax.set_ylabel("Number of successful attacks")
        ax.set_title(f"{model} {attack_type}-pixel attack")

    plt.suptitle(f"{model} - Original and Target Classes")
    plt.tight_layout()
    plt.show()

def plot_class_pairs_heatmap(results, model):
    # create three heatmaps 
    fig, axes = plt.subplots(1, 3, figsize=(18,6))

    # class labels 0-9
    classes = range(10)

    # Find common colour scale across all three heatmaps
    max_count = max(
        np.zeros((10, 10)).max(),
        *[
            np.histogram2d(
                values["org_cls_num"],
                values["tgt_cls_num"],
                bins=np.arange(11) - 0.5
            )[0].max()
            for values in results.values()
        ]
    )

    for ax, (attack_type, values) in zip(axes, results.items()):

        org_cls = values["org_cls_num"]
        tgt_cls = values["tgt_cls_num"]

        # initialise 10 x 10 matrix
        heatmap_data = np.zeros((10, 10), dtype=int)

        # count original-target class pairs
        for org, tgt in zip(org_cls, tgt_cls):
            heatmap_data[org, tgt] += 1

        # plot heatmap
        sns.heatmap(
            heatmap_data,
            annot=True,
            fmt="d",
            cmap="YlOrRd",
            vmin=0,
            vmax=max_count,
            xticklabels=classes,
            yticklabels=classes,
            cbar=True,
            ax=ax
        )

        # Colour axis labels
        ax.tick_params(axis="y", colors="red")
        ax.tick_params(axis="x", colors="blue")

        ax.set_xlabel("Target class", color="blue")
        ax.set_ylabel("Original class", color="red")

        ax.set_title(f"{model} {attack_type}-pixel attack")

    plt.suptitle(
        f"{model} - Original vs Target Class Distribution",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()


# visualises graphs for each model separately
for a in ["targeted"]:
    for m in ["VGG16", "AllConv", "NiN"]:
        results = calc_classes(a,m)
        plot_original_target_classes(results, m)
        plot_class_pairs_heatmap(results, m)
